### Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math


#Sklearn library
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate

###xgboost
from xgboost import XGBClassifier

### Overesample
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

import scipy.stats as stats

### SHAP Analysis
import shap

### COX
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter

from sklearn.feature_selection import VarianceThreshold

### Functions

In [ ]:
def age_group(age):
    if age >= 10 and age <= 20:
        return 1
    elif age > 20 and age <= 30:
        return 2
    elif age > 30 and age <= 40:
        return 3
    elif age > 40 and age <= 50:
        return 4
    elif age > 50 and age <= 60:
        return 5
    elif age > 60 and age <= 70:
        return 6
    elif age >70 and age <= 80:
        return 7
    else:
        return 8

### Read in data

In [ ]:
### Import data
input_Data = pd.read_csv('./input_data/one_hot_encoded_sum_per_pat.csv')
patient_data = pd.read_csv('./input_data/patient level data.csv')

In [ ]:
### add together columns of thiamine and thiamine hcl and remove thiamine hcl column if it is there
if '_thiamine hcl' in input_Data.columns:
    input_Data['_thiamine'] = input_Data['_thiamine'] + input_Data['_thiamine hcl']
    input_Data = input_Data.drop(columns=['_thiamine hcl'])

### add together columns of _hepari and _heparin and remove _heparin column if it is there
if '_hepari' in input_Data.columns:
    input_Data['_heparin'] = input_Data['_heparin'] + input_Data['_hepari']
    input_Data = input_Data.drop(columns=['_hepari'])

In [ ]:
### COX specific data ###############
patient_data['OUTCOME_DATE'] = pd.to_datetime(patient_data['OUTCOME_DATE'])
patient_data['HNC_DIAG_DATE'] = pd.to_datetime(patient_data['HNC_DIAG_DATE'])
patient_data['Survival_Time'] = (patient_data['OUTCOME_DATE'] - patient_data['HNC_DIAG_DATE']).dt.days
patient_data['event'] = patient_data['METASTASIS'].astype(int)

patient_data['AGE_GROUP'] = [age_group(x) for x in patient_data['AGE_AT_DIAGNOSIS']]
first_columns =['PATID','HNC_DIAG_DATE','OUTCOME_DATE','Survival_Time','event','METASTASIS', 'AGE_AT_DIAGNOSIS', 'AGE_GROUP']
other_columns = [col for col in patient_data.columns if col not in first_columns]
if 'Unnamed: 0' in other_columns:
    other_columns.remove('Unnamed: 0')
if 'index' in other_columns:
    other_columns.remove('index')
patient_data = patient_data[first_columns+other_columns]

In [ ]:
working_data = input_Data.merge(patient_data, on = ['PATID'], how = 'left')

In [ ]:
pat_data_columns = list(patient_data.columns)
pat_data_columns = pat_data_columns + ['CHEMOTHERAPY', 'IMMUNOTHERAPY', 'RADIATION', 'SURGERY']
other_columns = [col for col in input_Data.columns if col != 'PATID' and col not in pat_data_columns]
order = pat_data_columns + other_columns
order.remove('Unnamed: 0')
working_data = working_data[order]
working_data = working_data.dropna(subset=['METASTASIS'])

In [ ]:
working_data = working_data[working_data['Survival_Time']>2]

In [ ]:
working_data

In [ ]:
non_parse_columns = ['PATID','HNC_DIAG_DATE','OUTCOME_DATE', 'AGE_AT_DIAGNOSIS', 'AGE_GROUP']
parsing_columns = [col for col in working_data.columns if col not in non_parse_columns]
for col in tqdm(parsing_columns):
    working_data[col] = working_data[col].apply(lambda x: 0 if math.isnan(x) else x)
    working_data[col] = working_data[col].astype(int)

In [ ]:
working_data

In [ ]:
# ### Stage and hpv data
# # STAGE data
# patient_stage_data = pd.read_csv('./input_data/all results patient stage.csv')
# patient_stage_data = patient_stage_data[['PATID','STAGE']]
# working_data_with_stage = working_data.merge(patient_stage_data, on = 'PATID', how = 'left')
# working_data_with_stage = working_data_with_stage[~working_data_with_stage['STAGE'].isnull()]
# working_data = working_data_with_stage

### HPV data
HPV_individual_count_results = pd.read_csv('/Volumes/Tanikella_Pradham_IRB22-1959/yixiang/HPV/individual_count_results.csv')


HPVPatList = list(set(HPV_individual_count_results[HPV_individual_count_results['Category_1']==True]['PCDM_PATID']))
patList = list(set(working_data['PATID']))
nonHPVPatList = [item for item in patList if item not in HPVPatList]
print(len(HPVPatList))
print(len(patList))
print(len(nonHPVPatList))

HPVPat = list(set(HPV_individual_count_results[HPV_individual_count_results['Category_1']==True]['PCDM_PATID']))
HPVStatusdict = {}
for pat in tqdm(patList):
    if pat in HPVPat:
        HPVStatusdict[pat] = 1
    else:
        HPVStatusdict[pat] = 0

HPV_stat = [HPVStatusdict[pat] for pat in working_data['PATID']]
working_data['HPV_STATUS'] = HPV_stat

In [ ]:
working_data

### Functions

In [ ]:
def load_data(data):
    x = data.iloc[:,5::]
    y = data['METASTASIS']
    return x,y

def calculation_confounding_effect(data, variable):
    """
    This function calculates the Pearson correlation coefficient. 
    This metric measures the linear relationship between two variables by calculating the covariance of the variables divided by the product of their standard deviations. 
    Essentially, it quantifies how closely your variables move together in a straight-line pattern.
    INPUTS:
    data: input data frame for calculation,
    variable: the variable being examined for confounding effects.

    OUTPUTS:
    - table of correlation coeffecients between the variable and all other variables in the data frame.
    - chart displaying the top 20 correlation coeffecients with bar labels
    
    Here's a quick breakdown:
    Pearson correlation coefficient:
    Range: -1 to 1
    Interpretation:
    1 indicates a perfect positive linear relationship,
    -1 indicates a perfect negative linear relationship,
    0 indicates no linear relationship.
    Usage considerations: It is most appropriate when the data is normally distributed and the relationship between variables is linear
    
    0.00 - 0.10	Negligible or Very Weak
    0.10 - 0.30	Weak
    0.30 - 0.50	Moderate
    0.50 - 0.70	Strong
    0.70 - 0.90	Very Strong
    0.90 - 1.00	Nearly Perfect
    """

    df,y = load_data(data)

    correlations = df.apply(lambda x: x.corr(df[variable]))
    correlations = pd.DataFrame({
        'Feature': correlations.index,
        'Correlation': correlations.values,
        'Absolute Correlations': correlations.abs().values
    })
    correlations = correlations.sort_values(by = 'Absolute Correlations', ascending=False)
    
    top_corr_vars = list(correlations.head(20)['Feature'])
    add_vars = ['METASTASIS', 'IMMUNOTHERAPY', 'CHEMOTHERAPY', 'RADIATION', 'SURGERY', 'AGE_GROUP', 'GENDER']
    for var in add_vars:
        top_corr_vars.append(var)
    top_corr = correlations[correlations['Feature'].isin(top_corr_vars)]

    top_corr = top_corr.sort_values(by = 'Correlation', ascending=True)
    plt.figure(figsize=(15, 6))
    bars = plt.barh(top_corr['Feature'], top_corr['Correlation'])
    plt.xlabel('Correlation Coefficient')
    plt.ylabel('Feature')
    plt.title(f'Correlation with {variable[1:]}')

    offset = .0025
    for bar in bars:
        value = bar.get_width()
        y_center = bar.get_y() + bar.get_height() / 2
        # For negative values, use 0 (the right end of the bar) as the base position.
        base_position = value if value > 0 else 0
        plt.text(base_position + offset, y_center, f'{value:.3f}', va='center', ha='left')

    plt.show()
    print(correlations.head(20))
    return correlations


In [ ]:
# Function to identify drugs without strong confounder correlations
def filter_drugs_by_confounder_correlation(data, drug_list, confounders, correlation_threshold=0.5):
    """
    Filter drugs that don't have high correlations (>threshold) with key confounders
    
    Returns:
    - DataFrame with drugs and their correlations with confounders
    - List of drugs without high confounder correlations
    """
    df, y = load_data(data)
    
    results = []
    
    for drug in drug_list:
        if drug not in df.columns:
            continue
        
        drug_corr_data = {'Drug': drug}
        max_confounder_corr = 0
        problematic_confounders = []
        
        # Check correlation with each confounder
        for confounder in confounders:
            if confounder in df.columns:
                corr_value = df[drug].corr(df[confounder])
                drug_corr_data[f'{confounder}_corr'] = corr_value
                
                if abs(corr_value) > correlation_threshold:
                    problematic_confounders.append(f"{confounder} ({corr_value:.3f})")
                    max_confounder_corr = max(max_confounder_corr, abs(corr_value))
        
        drug_corr_data['Max_Confounder_Corr'] = max_confounder_corr
        drug_corr_data['High_Confounder_Corr'] = '; '.join(problematic_confounders) if problematic_confounders else 'None'
        drug_corr_data['Passes_Filter'] = len(problematic_confounders) == 0
        
        results.append(drug_corr_data)
    
    results_df = pd.DataFrame(results)
    filtered_drugs = results_df[results_df['Passes_Filter']]['Drug'].tolist()

    print(f"Originally {len(drug_list)} drugs")
    print(f"NOw {len(filtered_drugs)} drugs")
    
    return results_df, filtered_drugs

print("Confounder correlation filter function defined")

### Residual Confounding Sensitivity Analysis

**Committee Concern**: Even correlations below 0.5 (e.g., 0.3) can introduce meaningful bias through residual confounding with surgery. This analysis quantifies potential bias from unmeasured imbalances.

In [ ]:
def calculate_residual_confounding_bias(surgery_or, imbalance_proportion):
    """
    Calculate potential bias from residual confounding with surgery.
    
    Formula: Bias_HR = exp[ln(OR_surgery) × imbalance]
    
    Example from committee member:
    - If surgery has OR = 0.5 (protective)
    - And there's 20% imbalance in surgery rates between drug groups
    - Expected bias = exp[ln(0.5) × 0.20] = exp[-0.69 × 0.20] = 0.87
    
    Parameters:
    -----------
    surgery_or : float
        Odds ratio for surgery on metastasis outcome
    imbalance_proportion : float
        Proportion difference in surgery rates (e.g., 0.20 = 20% difference)
    
    Returns:
    --------
    bias_hr : float
        Expected bias in observed HR due to confounding
    """
    import numpy as np
    ln_or = np.log(surgery_or)
    bias_ln_hr = ln_or * imbalance_proportion
    bias_hr = np.exp(bias_ln_hr)
    return bias_hr


def residual_confounding_sensitivity_analysis(data, drug_list, confounder='SURGERY', 
                                              imbalance_levels=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30]):
    """
    Perform sensitivity analysis for residual confounding with surgery.
    
    This addresses the committee concern that even low correlations (0.3) could 
    yield HRs as large as those observed if unmeasured imbalances exist.
    
    Returns: DataFrame with potential bias at different imbalance levels
    """
    import numpy as np
    import pandas as pd
    
    df, y = load_data(data)
    
    # Calculate surgery OR from data
    if confounder not in df.columns:
        print(f"Error: {confounder} not found in data")
        return None
    
    surgery_exposed = df[confounder] == 1
    met_with_surgery = y[surgery_exposed].sum()
    no_met_with_surgery = (~y[surgery_exposed].astype(bool)).sum()
    met_without_surgery = y[~surgery_exposed].sum()
    no_met_without_surgery = (~y[~surgery_exposed].astype(bool)).sum()
    
    or_surgery = (met_with_surgery * no_met_without_surgery) / (met_without_surgery * no_met_with_surgery)
    ln_or_surgery = np.log(or_surgery)
    
    print("=" * 110)
    print("RESIDUAL CONFOUNDING SENSITIVITY ANALYSIS")
    print("=" * 110)
    print("\nCommittee Concern: 'Correlations of 0.3 could very easily yield HRs as large")
    print("as those observed. Residual confounding with surgery is a genuine possibility.'")
    
    print(f"\n{'='*110}")
    print(f"SURGERY EFFECT IN DATA:")
    print(f"{'='*110}")
    print(f"  Odds Ratio (Surgery → Metastasis): {or_surgery:.4f}")
    print(f"  Log-Odds Ratio: {ln_or_surgery:.4f}")
    if or_surgery > 1:
        print(f"  → Surgery shows INCREASED metastasis risk (confounding by indication)")
    else:
        print(f"  → Surgery shows DECREASED metastasis risk")
    
    print(f"\n{'='*110}")
    print(f"POTENTIAL BIAS FROM UNMEASURED SURGERY IMBALANCE:")
    print(f"{'='*110}")
    print(f"\nIf drug-exposed patients differ in surgery rates from unexposed:")
    print(f"\n{'Imbalance':>12} | {'Bias HR':>10} | {'Effect on Observed HR'}")
    print("-" * 110)
    
    sensitivity_data = []
    for imbalance in imbalance_levels:
        bias_hr = calculate_residual_confounding_bias(or_surgery, imbalance)
        
        if bias_hr < 1:
            direction = f"Makes drugs appear {(1-bias_hr)*100:.1f}% MORE protective"
        else:
            direction = f"Makes drugs appear {(bias_hr-1)*100:.1f}% LESS protective"
        
        print(f"{imbalance*100:>10.0f}%  | {bias_hr:>10.4f} | {direction}")
        
        sensitivity_data.append({
            'Imbalance_Percent': imbalance * 100,
            'Bias_HR': bias_hr,
            'Bias_Percent': (bias_hr - 1) * 100
        })
    
    sensitivity_df = pd.DataFrame(sensitivity_data)
    
    # Committee member's example
    print(f"\n{'='*110}")
    print(f"COMMITTEE MEMBER'S EXAMPLE:")
    print(f"{'='*110}")
    print(f"Theoretical: If surgery OR = 0.5 and 20% imbalance:")
    print(f"  → Bias = exp[ln(0.5) × 0.20] = exp[-0.69 × 0.20] = 0.87")
    print(f"\nUsing OUR surgery OR = {or_surgery:.4f} and 20% imbalance:")
    bias_20pct = calculate_residual_confounding_bias(or_surgery, 0.20)
    print(f"  → Bias = exp[ln({or_surgery:.4f}) × 0.20] = {bias_20pct:.4f}")
    
    # Check actual drug-surgery correlations
    print(f"\n{'='*110}")
    print(f"DRUGS WITH NOTABLE SURGERY CORRELATIONS:")
    print(f"{'='*110}")
    print(f"\n{'Drug':<45} | {'r(Surgery)':>12} | {'Concern Level'}")
    print("-" * 110)
    
    concern_drugs = []
    for drug in drug_list[:30]:
        if drug not in df.columns:
            continue
        corr = df[drug].corr(df[confounder])
        
        if abs(corr) >= 0.30:
            concern = "⚠️  May introduce meaningful bias"
            concern_drugs.append((drug, corr))
            print(f"{drug:<45} | {corr:>12.3f} | {concern}")
        elif abs(corr) >= 0.20:
            concern = "⚡ Monitor - possible minor bias"
            print(f"{drug:<45} | {corr:>12.3f} | {concern}")
    
    print(f"\n{'='*110}")
    print(f"FOR DISSERTATION - LIMITATION SECTION:")
    print(f"{'='*110}")
    print(f"\nSUGGESTED TEXT:")
    print(f"'While we adjusted for surgery in our models, residual confounding remains")
    print(f"a genuine possibility. As demonstrated in sensitivity analysis, correlations")
    print(f"as low as 0.3 between medications and unmeasured disease severity indicators")
    print(f"could yield hazard ratios as large as those observed. Using the observed")
    print(f"surgery effect (OR={or_surgery:.2f}) and assuming a 20% imbalance in surgery")
    print(f"eligibility between medication-exposed and unexposed groups, the expected")
    print(f"bias would be HR={bias_20pct:.2f}. These findings will require experimental")
    print(f"validation to establish causal relationships beyond the observed correlations.'")
    
    return sensitivity_df


print("✓ Residual confounding sensitivity analysis functions defined")

In [ ]:
def display_drug_confounder_correlations(data, drug_list, confounders=['SURGERY', 'RADIATION', 'CHEMOTHERAPY', 'IMMUNOTHERAPY', 'AGE_GROUP', 'GENDER']):
    """
    Display correlation values between drugs and treatment confounders.
    Parameters:
    -----------
    data : DataFrame
        Patient-level data with drug and confounder columns
    drug_list : list
        List of drug feature names to analyze
    confounders : list
        List of confounder column names

    Returns:
    --------
    corr_df : DataFrame
        Correlation matrix with drugs as rows, confounders as columns
    """
    present_confounders = [conf for conf in confounders if conf in data.columns]
    if not present_confounders:
        print("No confounder columns found in the data.")
        return pd.DataFrame(), pd.DataFrame()
    corr_dict = {}
    for drug in drug_list:
        if drug in data.columns:
            corr_dict[drug] = {}
            for conf in present_confounders:
                corr_dict[drug][conf] = data[drug].corr(data[conf])
    corr_df = pd.DataFrame(corr_dict).T[present_confounders]
    display(corr_df)
    return corr_df


In [ ]:
def analyze_drug_to_all_feature_correlations(data, drug_list, hpv_status='pos', threshold=0.5, save_outputs=True):
    """
    Analyze correlations between top drugs and ALL features in the dataset.
    Shows which features (drugs, ICD codes, CPT codes, etc.) correlate with each medication.
    
    Parameters:
    -----------
    data : DataFrame
        Patient-level data with all features
    drug_list : list
        List of drug names to analyze (in priority order)
    hpv_status : str
        'pos' or 'neg' for file naming
    threshold : float
        Correlation threshold for identifying high correlations (default 0.5)
    top_n_drugs : int
        Number of top drugs to analyze
    save_outputs : bool
        Whether to save figures and tables
    
    Returns:
    --------
    drug_correlations_df : DataFrame
        All drug-feature pairs with correlations > threshold
    drug_summary_df : DataFrame
        Summary for each drug showing count and list of correlated features
    """
    import seaborn as sns
    
    # Get data features
    df, y = load_data(data)
    all_feature_names = list(df.columns)
    
    # Select top drugs that exist in data
    top_drugs = [d for d in drug_list if d in df.columns]
    
    if len(top_drugs) == 0:
        print("No drugs found in dataset.")
        return pd.DataFrame(), pd.DataFrame()
    
    print(f"\n{'='*120}")
    print(f"HPV{hpv_status.upper()} DRUG-TO-ALL-FEATURE CORRELATION ANALYSIS")
    print(f"{'='*120}")
    print(f"\nAnalyzing {len(top_drugs)} top drugs against {len(all_feature_names)} total features")
    print(f"Correlation threshold: |r| > {threshold}")
    
    # =========================================================================
    # Calculate correlations for each drug against all features
    # =========================================================================
    all_drug_feature_pairs = []
    drug_summary_data = []
    
    for drug in top_drugs:
        # Calculate correlations with all features
        correlations = df.apply(lambda x: x.corr(df[drug]))
        
        # Find features with high correlations (excluding the drug itself)
        high_corr_features = []
        for feature, corr_val in correlations.items():
            if feature != drug and abs(corr_val) > threshold:
                all_drug_feature_pairs.append({
                    'Drug': drug,
                    'Correlated_Feature': feature,
                    'Correlation': corr_val,
                    'Abs_Correlation': abs(corr_val)
                })
                high_corr_features.append(f"{feature} ({corr_val:.3f})")
        
        # Create summary for this drug
        drug_summary_data.append({
            'Drug': drug,
            'Num_High_Correlations': len(high_corr_features),
            'Correlated_Features': '; '.join(high_corr_features) if high_corr_features else 'None'
        })
    
    # Create DataFrames
    drug_correlations_df = pd.DataFrame(all_drug_feature_pairs)
    drug_summary_df = pd.DataFrame(drug_summary_data)
    
    # Sort by number of correlations
    drug_summary_df = drug_summary_df.sort_values('Num_High_Correlations', ascending=False)
    
    print(f"\n{'='*120}")
    print(f"RESULTS SUMMARY:")
    print(f"{'='*120}")
    print(f"\nTotal drug-feature pairs with |r| > {threshold}: {len(drug_correlations_df)}")
    print(f"Drugs with at least one high correlation: {len(drug_summary_df[drug_summary_df['Num_High_Correlations'] > 0])}")
    print(f"Drugs with no high correlations: {len(drug_summary_df[drug_summary_df['Num_High_Correlations'] == 0])}")
    
    # =========================================================================
    # TABLE 1: Drug Summary Table
    # =========================================================================
    print(f"\n{'='*120}")
    print(f"DRUG CORRELATION SUMMARY (Top Drugs):")
    print(f"{'='*120}\n")
    print(f"{'Drug':<40} | {'# Correlated':>13} | {'Top Correlated Features'}")
    print(f"{'-'*120}")
    for idx, row in drug_summary_df.iterrows():
        corr_preview = row['Correlated_Features'] + '...' if len(row['Correlated_Features']) > 65 else row['Correlated_Features']
        print(f"{row['Drug']:<40} | {row['Num_High_Correlations']:>13} | {corr_preview}")
    
    # if save_outputs:
    #     drug_summary_df.to_csv(f'./Results/hpv_{hpv_status}_drug_correlation_summary.csv', index=False)
    #     print(f"\n✓ Drug summary saved: ./Results/hpv_{hpv_status}_drug_correlation_summary.csv")
    
    # =========================================================================
    # TABLE 2: Detailed Drug-Feature Pairs
    # =========================================================================
    if len(drug_correlations_df) > 0:
        drug_correlations_df = drug_correlations_df.sort_values('Abs_Correlation', ascending=False)
        
        print(f"\n{'='*120}")
        print(f"TOP DRUG-FEATURE CORRELATION PAIRS:")
        print(f"{'='*120}\n")
        print(f"{'Drug':<40} | {'Correlated Feature':<40} | {'Correlation':>12}")
        print(f"{'-'*120}")
        for idx, row in drug_correlations_df.iterrows():
            print(f"{row['Drug']:<40} | {row['Correlated_Feature']:<40} | {row['Correlation']:>12.3f}")
        
        # if save_outputs:
        #     drug_correlations_df.to_csv(f'./Results/hpv_{hpv_status}_drug_feature_correlations.csv', index=False)
        #     print(f"\n✓ Detailed correlations saved: ./Results/hpv_{hpv_status}_drug_feature_correlations.csv")
        #     print(f"  (Total pairs: {len(drug_correlations_df)})")
    
    # =========================================================================
    # VISUALIZATION 1: Bar Chart - Drugs by Number of Correlations
    # =========================================================================
    fig, ax = plt.subplots(figsize=(14, 10))
    
    plot_data = drug_summary_df
    colors = ['steelblue' if x > 0 else 'lightgray' for x in plot_data['Num_High_Correlations'].values]
    
    bars = ax.barh(range(len(plot_data)), plot_data['Num_High_Correlations'].values,
                   color=colors, edgecolor='black', linewidth=0.5)
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, plot_data['Num_High_Correlations'].values)):
        if val > 0:
            ax.text(val + 0.5, i, str(int(val)), va='center', ha='left', 
                   fontsize=9, fontweight='bold')
    
    ax.set_yticks(range(len(plot_data)))
    ax.set_yticklabels(plot_data['Drug'].values, fontsize=10)
    ax.set_xlabel(f'Number of Features with |r| > {threshold}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Drug', fontsize=12, fontweight='bold')
    ax.set_title(f'HPV{hpv_status.upper()} Top Drugs: Number of High Correlations with All Features', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    if len(plot_data[plot_data['Num_High_Correlations'] > 0]) > 0:
        max_val = max(plot_data['Num_High_Correlations'].values)
        ax.set_xlim(0, max_val * 1.15)
    
    plt.tight_layout()
    if save_outputs:
        plt.savefig(f'./Results/hpv_{hpv_status}_drug_correlation_counts.png', 
                    dpi=300, bbox_inches='tight')
        print(f"\n✓ Bar chart saved: ./Results/hpv_{hpv_status}_drug_correlation_counts.png")
    plt.show()
    
    # =========================================================================
    # VISUALIZATION 2: Individual Drug Correlation Bar Charts (Stacked)
    # =========================================================================
    print(f"\n{'='*120}")
    print(f"CREATING INDIVIDUAL DRUG CORRELATION VISUALIZATIONS (|r| >= {threshold})...")
    print(f"{'='*120}")
    
    # Collect data for each drug showing features with |r| >= threshold
    drugs_with_correlations = []
    for drug in top_drugs:
        correlations = df.apply(lambda x: x.corr(df[drug]))
        
        # Find features with |r| >= threshold (excluding the drug itself)
        high_corr_features = []
        for feature, corr_val in correlations.items():
            if feature != drug and abs(corr_val) >= threshold:
                high_corr_features.append({
                    'Feature': feature,
                    'Correlation': corr_val,
                    'Abs_Correlation': abs(corr_val)
                })
        
        if len(high_corr_features) > 0:
            drugs_with_correlations.append({
                'Drug': drug,
                'Correlations': pd.DataFrame(high_corr_features).sort_values('Abs_Correlation', ascending=False)
            })
    
    print(f"\nFound {len(drugs_with_correlations)} drugs with at least one feature correlation >= {threshold}")
    
    if len(drugs_with_correlations) > 0:
        # Create individual figure for each drug with dynamic sizing
        for idx, drug_data in enumerate(drugs_with_correlations, 1):
            drug_name = drug_data['Drug']
            corr_df = drug_data['Correlations']
            
            # Sort by correlation value (not absolute) for better visualization
            corr_df_sorted = corr_df.sort_values('Correlation', ascending=True)
            
            # Dynamic figure height based on number of features (0.3 inches per feature, min 4, max 30)
            n_features = len(corr_df_sorted)
            fig_height = max(4, min(30, n_features * 0.3))
            fig_width = 14
            
            # Dynamic font size based on number of features
            y_fontsize = max(7, min(10, 150 // n_features))
            
            # Create figure
            fig, ax = plt.subplots(figsize=(fig_width, fig_height))
            
            # Color code: positive = blue, negative = red
            colors = ['#d73027' if x < 0 else '#4575b4' for x in corr_df_sorted['Correlation']]
            
            # Create horizontal bar chart
            bars = ax.barh(range(len(corr_df_sorted)), 
                          corr_df_sorted['Correlation'].values,
                          color=colors, 
                          edgecolor='black', 
                          linewidth=0.7,
                          alpha=0.8)
            
            # Add value labels
            for i, (bar, val) in enumerate(zip(bars, corr_df_sorted['Correlation'].values)):
                label_x = val + (0.02 if val > 0 else -0.02)
                ha = 'left' if val > 0 else 'right'
                ax.text(label_x, i, f'{val:.3f}', 
                       va='center', ha=ha, fontsize=8, fontweight='bold')
            
            # Set y-axis labels (feature names)
            ax.set_yticks(range(len(corr_df_sorted)))
            ax.set_yticklabels(corr_df_sorted['Feature'].values, fontsize=y_fontsize)
            
            # Formatting
            ax.set_xlabel('Correlation Coefficient', fontsize=11, fontweight='bold')
            clean_drug_name = drug_name[1:] if drug_name.startswith("_") else drug_name
            ax.set_title(f'HPV{hpv_status.upper()}: {clean_drug_name} - Feature Correlations (|r| >= {threshold})', 
                        fontsize=12, fontweight='bold', pad=15)
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1.2, alpha=0.5)
            ax.set_xlim(-1.05, 1.05)
            ax.grid(axis='x', alpha=0.3, linestyle='--')
            
            # Add count annotation
            ax.text(0.98, 0.98, f'n={len(corr_df_sorted)} features', 
                   transform=ax.transAxes, fontsize=10, 
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))
            
            plt.tight_layout()
            
            if save_outputs:
                # Save each drug's figure separately
                safe_drug_name = clean_drug_name.replace(' ', '_').replace('/', '_')
                filename = f'./Results/hpv_{hpv_status}_correlation_{safe_drug_name}.png'
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                print(f"\n✓ [{idx}/{len(drugs_with_correlations)}] Saved: {filename}")
                print(f"    {clean_drug_name}: {n_features} correlated features | Figure size: {fig_width}\" × {fig_height:.1f}\"")
            
            plt.show()
        
        print(f"\n{'='*120}")
        print(f"✓ All {len(drugs_with_correlations)} drug correlation charts created and saved individually")
        print(f"  Threshold used: |r| >= {threshold}")
        print(f"  Files saved to: ./Results/hpv_{hpv_status}_correlation_[drug_name].png")
    else:
        print(f"\n⚠ No drugs found with feature correlations >= {threshold}")
    
    # =========================================================================
    # INTERPRETATION
    # =========================================================================
    print(f"\n{'='*120}")
    print(f"INTERPRETATION:")
    print(f"{'='*120}")
    
    drugs_with_corr = drug_summary_df[drug_summary_df['Num_High_Correlations'] > 0]
    drugs_without_corr = drug_summary_df[drug_summary_df['Num_High_Correlations'] == 0]
    
    if len(drugs_with_corr) == 0:
        print(f"\n✓ EXCELLENT: No drugs show high correlations (|r| > {threshold}) with other features")
        print(f"  → Minimal multicollinearity detected")
        print(f"  → Drugs appear to represent independent therapeutic signals")
    else:
        print(f"\n⚠ FINDINGS:")
        print(f"  • {len(drugs_with_corr)} drugs have ≥1 high correlation with other features")
        print(f"  • {len(drugs_without_corr)} drugs have no high correlations")
        print(f"  • Total drug-feature pairs with |r| > {threshold}: {len(drug_correlations_df)}")
        print(f"\n  NEXT STEPS:")
        print(f"  → Review correlated features to understand co-prescription patterns")
        print(f"  → Consider these correlations when interpreting drug effects")
        print(f"  → High correlations may indicate similar mechanisms or shared indications")
    
    return drug_correlations_df, drug_summary_df

print("Drug-to-all-feature correlation analysis function defined successfully!")

In [ ]:
# Function to create correlation summary table for top features
def summarize_high_correlations(data, features, correlation_threshold=0.5):
    """
    For each feature, identify which features correlate above the threshold
    
    Returns:
    - DataFrame with feature and list of highly correlated features (>threshold)
    """
    df, y = load_data(data)
    
    summary_data = []
    
    for feature in features:
        if feature not in df.columns:
            continue
            
        # Calculate correlations
        correlations = df.apply(lambda x: x.corr(df[feature]))
        correlations_df = pd.DataFrame({
            'Feature': correlations.index,
            'Correlation': correlations.values,
            'Absolute_Correlation': correlations.abs().values
        })
        
        # Find features with correlation > threshold (excluding the feature itself)
        high_corr = correlations_df[
            (correlations_df['Absolute_Correlation'] > correlation_threshold) & 
            (correlations_df['Feature'] != feature)
        ].sort_values('Absolute_Correlation', ascending=False).head(20)
        
        # Format high correlation features
        high_corr_list = [f"{row['Feature']} ({row['Correlation']:.3f})" 
                         for _, row in high_corr.iterrows()]
        
        summary_data.append({
            'Feature': feature,
            'Num_High_Correlations': len(high_corr_list),
            'High_Correlations_Above_0.5': '; '.join(high_corr_list) if high_corr_list else 'None'
        })
    
    return pd.DataFrame(summary_data)

print("Correlation summary function defined")

In [ ]:
def load_data(data):
    x = data.iloc[:,7::]
    x = x.fillna(0)
    x = x.astype(int)
    y = data['METASTASIS']
    y = y.fillna(0)
    y = y.astype(int)
    return x,y

In [ ]:
# Helper function to extract drug names (non-confounders, non-ICD codes)
def pull_drugs(feature_list, has_underscore = True):
    """Function to pull the top drugs from the feature importance dataframe. As some features are not drugs need to filter to only drugs."""
    ### Pull the top features based on xgb importance
    ### check if the feature is a drug and not a procedure code or something else
    ### features could be: '_Z85.3','_Z08','_J96.01','IMMUNOTHERAPY','_insulin lispro',
    #### need to pull only drugs
    if has_underscore:
        start_position = 1
    else:
        start_position = 0
     ### if the second character is a digit then it is not a drug
    top_drugs = []
    for feature in feature_list:
        if feature[start_position+1].isdigit():
            continue
        else:
            top_drugs.append(feature)
    return top_drugs

print("Drug extraction helper function defined")

### Import data

In [ ]:
# Load ML feature importance results
import os
print("=" * 100)
print("LOADING COX AND ML RESULTS FOR VALIDATION")
print("=" * 100)
# Check if files exist
required_files = [
    './Results/ML analysis/hpv_positive_ml_xgb_results.csv',
    './Results/ML analysis/hpv_negative_ml_xgb_results.csv',
]

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print("\n⚠️ WARNING: The following required files are missing:")
    for f in missing_files:
        print(f"   - {f}")
    print("\n📋 ACTION REQUIRED:")
    print("   1. Run '02 data_analysis_ml.ipynb' to generate ML results")
    print("   2. Run '02_data_analysis_cox new.ipynb' to generate Cox results")
    print("   3. Then re-run this notebook")
    raise FileNotFoundError("Required result files not found. Please run the ML and Cox analysis notebooks first.")


# Load the files
ml_hpv_pos = pd.read_csv('./Results/ML analysis/hpv_positive_ml_xgb_results.csv')
ml_hpv_neg = pd.read_csv('./Results/ML analysis/hpv_negative_ml_xgb_results.csv')
# cox_hpv_pos = pd.read_csv('./Results/cox_comprehensive_hpv_positive.csv')
# cox_hpv_neg = pd.read_csv('./Results/cox_comprehensive_hpv_negative.csv')

print("\n✓ All files loaded successfully!")
print("\nML Feature Importance Tables Loaded:")
print(f"  HPV+ ML features: {len(ml_hpv_pos)}")
print(f"  HPV- ML features: {len(ml_hpv_neg)}")

# print("\nCox Analysis Tables Loaded (Comprehensive - Univariate + Multivariate):")
# print(f"  HPV+ Cox features: {len(cox_hpv_pos)}")
# print(f"  HPV- Cox features: {len(cox_hpv_neg)}")

# # Display column info
# print(f"\nCox HPV+ columns: {list(cox_hpv_pos.columns)[:5]}... ({len(cox_hpv_pos.columns)} total)")
print(f"ML HPV+ columns: {list(ml_hpv_pos.columns)[:5]}... ({len(ml_hpv_pos.columns)} total)")

In [ ]:
### filter to just drugs 
hpv_pos_drugs = pull_drugs(ml_hpv_pos['feature'])
hpv_neg_drugs = pull_drugs(ml_hpv_neg['feature'])

### HPV+

In [ ]:
hpv_positive_working_data = working_data[working_data['HPV_STATUS']==1]
hpv_positive_working_data = hpv_positive_working_data.drop(columns = ['HPV_STATUS'])
len(hpv_positive_working_data)

#### Extract Top Drugs from ML Analysis

In [ ]:
# Extract top drug features from ML analysis for both subgroups
print("=" * 100)
print("TOP DRUGS FROM ML ANALYSIS (XGBoost Importance)")
print("=" * 100)

# HPV+ top drugs
ml_hpv_pos_sorted = ml_hpv_pos.sort_values('xgb_absolute_Importance', ascending=False)
top_features_hpv_pos = ml_hpv_pos_sorted['feature'].tolist()
top_hpv_pos_meds = pull_drugs(top_features_hpv_pos)

print(f"\nHPV+ Top Drugs:")
for i, drug in enumerate(top_hpv_pos_meds, 1):
    drug_data = ml_hpv_pos[ml_hpv_pos['feature'] == drug].iloc[0]
    print(f"{i:2d}. {drug:45s} | XGB Importance: {drug_data['xgb_absolute_Importance']:.4f} | "
          f"SHAP: {drug_data['XGB Mean Absolute SHAP Value']:.4f}")

In [ ]:
### Print correlations
ml_hpv_pos_sorted = ml_hpv_pos.sort_values('xgb_absolute_Importance', ascending=False)
top_features_hpv_pos = ml_hpv_pos_sorted['feature'].tolist()
top_hpv_pos_meds = pull_drugs(top_features_hpv_pos)

for var in top_hpv_pos_meds:
    calculation_confounding_effect(hpv_positive_working_data, var)

#### HPV+ Drugs Without Strong Confounder Correlations

In [ ]:
# Define key confounders
key_confounders = ['SURGERY', 'RADIATION', 'CHEMOTHERAPY', 'IMMUNOTHERAPY']

# Filter HPV+ drugs by confounder correlation
print("=" * 120)
print("HPV+ TOP DRUGS - CONFOUNDER CORRELATION ANALYSIS")
print("=" * 120)

hpv_pos_confounder_analysis, hpv_pos_filtered_drugs = filter_drugs_by_confounder_correlation(
    hpv_positive_working_data,
    top_hpv_pos_meds,
    key_confounders,
    correlation_threshold=0.5
)

# Display results
print(f"\nAnalyzing correlations with confounders: {', '.join(key_confounders)}")
print(f"Correlation threshold: >0.5\n")

print(f"{'Drug':<45} | {'Max Conf Corr':>14} | {'Status':>12} | {'Problematic Confounders'}")
print("-" * 120)

for idx, row in hpv_pos_confounder_analysis.iterrows():
    status = "✓ PASS" if row['Passes_Filter'] else "✗ FAIL"
    print(f"{row['Drug']:<45} | {row['Max_Confounder_Corr']:>14.3f} | {status:>12} | {row['High_Confounder_Corr']}")

print(f"\n{'='*120}")
print(f"SUMMARY:")
print(f"  Total drugs analyzed: {len(hpv_pos_confounder_analysis)}")
print(f"  Drugs WITHOUT strong confounder correlations (>0.5): {len(hpv_pos_filtered_drugs)}")
print(f"  Drugs WITH strong confounder correlations: {len(hpv_pos_confounder_analysis) - len(hpv_pos_filtered_drugs)}")

print(f"\n✓ FILTERED DRUGS (no strong confounder correlations):")
for i, drug in enumerate(hpv_pos_filtered_drugs, 1):
    print(f"  {i:2d}. {drug}")

# # Save results
# hpv_pos_confounder_analysis.to_csv('./Results/hpv_positive_top25_confounder_analysis.csv', index=False)
# print(f"\n✓ Saved to: ./Results/hpv_positive_top25_confounder_analysis.csv")

#### Visualization: Drug-Confounder Correlations (HPV+)

In [ ]:
# Analyze drug-confounder correlations for HPV+ cohort
hpv_pos_data = working_data[working_data['HPV_STATUS'] == 1].copy()
top_drugs_hpv_pos = pull_drugs(ml_hpv_pos['feature'].to_list()) # Top 30 drugs
confounders = ['SURGERY', 'RADIATION', 'CHEMOTHERAPY', 'IMMUNOTHERAPY']
top_drugs_hpv_pos = [d for d in top_drugs_hpv_pos if d not in confounders]

# present_confounders = [conf for conf in confounders if conf in data.columns]
# if not present_confounders:
#     print("No confounder columns found in the data.")
#     return pd.DataFrame(), pd.DataFrame()
# corr_dict = {}
# for drug in drug_list:
#     if drug in data.columns:
#         corr_dict[drug] = {}
#         for conf in present_confounders:
#             corr_dict[drug][conf] = data[drug].corr(data[conf])
# corr_df = pd.DataFrame(corr_dict).T[present_confounders]
# display(corr_df)
# return corr_df

drug_confounder_corr_pos = display_drug_confounder_correlations(
    data=hpv_pos_data,
    drug_list=top_drugs_hpv_pos,
)

In [ ]:
drug_confounder_corr_pos[['AGE_GROUP', 'GENDER']]

#### Residual Confounding Sensitivity Analysis (HPV+)

Calculate potential bias from unmeasured surgery imbalances.

In [ ]:
# Run residual confounding sensitivity analysis for HPV+ cohort
sensitivity_hpv_pos = residual_confounding_sensitivity_analysis(
    data=hpv_pos_data,
    drug_list=top_drugs_hpv_pos,
    confounder='SURGERY',
    imbalance_levels=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
)

In [ ]:
# Display sensitivity table
print("\nSENSITIVITY ANALYSIS RESULTS:")
print(sensitivity_hpv_pos)

#### Feature-to-Feature Correlations >0.5 (HPV+)

In [ ]:
# Analyze correlations between top HPV+ drugs and ALL features in dataset
drug_correlations_pos, drug_summary_pos = analyze_drug_to_all_feature_correlations(
    data=hpv_pos_data,
    drug_list=top_drugs_hpv_pos,
    hpv_status='pos',
    threshold=0.5,
    save_outputs=True
)

### Residual Confounding - Quick Example for Dissertation

Run these cells to generate dissertation text with your actual data:

In [ ]:
# Example: Run sensitivity analysis for HPV+ top drugs
# (Uses the functions defined earlier in the notebook)

df_pos, y_pos = load_data(hpv_pos_data)

# Get surgery OR from your actual data
surgery_exposed = df_pos['SURGERY'] == 1
met_with_surgery = y_pos[surgery_exposed].sum()
no_met_with_surgery = (~y_pos[surgery_exposed].astype(bool)).sum()
met_without_surgery = y_pos[~surgery_exposed].sum()
no_met_without_surgery = (~y_pos[~surgery_exposed].astype(bool)).sum()

or_surgery = (met_with_surgery * no_met_without_surgery) / (met_without_surgery * no_met_with_surgery)

print(f"Surgery OR from your data (HPV+): {or_surgery:.4f}")
print(f"\nUsing the calculate_residual_confounding_bias() function:")

# Example: 20% imbalance scenario
bias_20pct = calculate_residual_confounding_bias(or_surgery, 0.20)
print(f"  20% imbalance → Bias HR = {bias_20pct:.4f}")

# Generate dissertation text
print("\n" + "="*80)
print("DISSERTATION TEXT (copy this):")
print("="*80)
print(f"""
While we adjusted for surgery in multivariable models, residual confounding 
remains a genuine possibility. As demonstrated through sensitivity analysis, 
correlations as low as 0.3 between medications and unmeasured disease severity 
indicators could yield hazard ratios as large as those observed. Using the 
observed surgery effect (OR={or_surgery:.2f}) and assuming a 20% imbalance in 
surgery eligibility between medication-exposed and unexposed groups—representing 
unmeasured differences in disease severity or clinical characteristics—the 
expected bias would be HR={bias_20pct:.2f}. This demonstrates that even modest 
unmeasured confounding could account for effect sizes comparable to our findings. 
Therefore, experimental validation will be required to establish causal 
relationships beyond the observed statistical associations.
""")

### HPV-

In [ ]:
# HPV- top drugs
print(f"\n{'='*100}")
print(f"HPV- Top Drugs:")
print(f"{'='*100}")

ml_hpv_neg_sorted = ml_hpv_neg.sort_values('xgb_absolute_Importance', ascending=False)
top_features_hpv_neg = ml_hpv_neg_sorted['feature'].tolist()
top_hpv_neg_meds = pull_drugs(top_features_hpv_neg)

print(f"\nHPV- Top Drugs:")
for i, drug in enumerate(top_hpv_neg_meds, 1):
    drug_data = ml_hpv_neg[ml_hpv_neg['feature'] == drug].iloc[0]
    print(f"{i:2d}. {drug:45s} | XGB Importance: {drug_data['xgb_absolute_Importance']:.4f} | "
          f"SHAP: {drug_data['XGB Mean Absolute SHAP Value']:.4f}")


In [ ]:
hpv_negative_working_data = working_data[working_data['HPV_STATUS']==0]
hpv_negative_working_data = hpv_negative_working_data.drop(columns = ['HPV_STATUS'])
len(hpv_negative_working_data)

In [ ]:
# Create correlation summary for top 25 HPV- features
print("=" * 120)
print("HPV- TOP FEATURES - CORRELATION SUMMARY (Features with >0.5 correlation)")
print("=" * 120)

hpv_neg_corr_summary = summarize_high_correlations(
    hpv_negative_working_data, 
    top_hpv_neg_meds, 
    correlation_threshold=0.5
)

# Display the summary
for idx, row in hpv_neg_corr_summary.iterrows():
    print(f"\n{idx+1}. {row['Feature']}")
    print(f"   Number of high correlations (>0.5): {row['Num_High_Correlations']}")
    if row['High_Correlations_Above_0.5'] != 'None':
        print(f"   High correlations: {row['High_Correlations_Above_0.5']}")
    else:
        print(f"   High correlations: None")

# Save to CSV
# hpv_neg_corr_summary.to_csv('./Results/hpv_negative_top25_correlation_summary.csv', index=False)
# print(f"\n✓ Saved to: ./Results/hpv_negative_top25_correlation_summary.csv")

#### Visualization: Drug-Confounder Correlations (HPV-)

In [ ]:
# Analyze drug-confounder correlations for HPV- cohort
hpv_neg_data = working_data[working_data['HPV_STATUS'] == 0].copy()
top_drugs_hpv_neg = pull_drugs(ml_hpv_neg['feature']) # Top 30 drugs
confounders = ['SURGERY', 'RADIATION', 'CHEMOTHERAPY', 'IMMUNOTHERAPY']
top_drugs_hpv_neg = [d for d in top_drugs_hpv_neg if d not in confounders]
drug_confounder_corr_neg = display_drug_confounder_correlations(
    data=hpv_neg_data,
    drug_list=top_drugs_hpv_neg,
)

In [ ]:
drug_confounder_corr_neg[['AGE_GROUP','GENDER']]

#### Residual Confounding Sensitivity Analysis (HPV-)

**Addressing Committee Concern**: Calculate potential bias from unmeasured surgery imbalances.

In [ ]:
# Run residual confounding sensitivity analysis for HPV- cohort
sensitivity_hpv_neg = residual_confounding_sensitivity_analysis(
    data=hpv_neg_data,
    drug_list=top_drugs_hpv_neg,
    confounder='SURGERY',
    imbalance_levels=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
)

In [ ]:
# Display sensitivity table
print("\nSENSITIVITY ANALYSIS RESULTS:")
print(sensitivity_hpv_neg)

#### Feature-to-Feature Correlations >0.5 (HPV-)

In [ ]:
# Analyze correlations between top HPV- drugs and ALL features in dataset
drug_correlations_neg, drug_summary_neg = analyze_drug_to_all_feature_correlations(
    data=hpv_neg_data,
    drug_list=top_drugs_hpv_neg,
    hpv_status='neg',
    threshold=0.5,
    save_outputs=True
)